# Real-time tracking with propygator

This notebook walks through Feature 1.4: answer *where is this satellite right now*, then
*show me* — a live, self-updating dashboard.

1.4 is mostly **composition** over Feature 1.3's `propagate_tle` and Feature 1.1's output
surface, so it adds little new physics. Three layers:

- **realtime primitives** — `current_state` / `current_ground_position`, cheap one-shot
  "where is it now" queries;
- the **observer sky view** — the `look_angles` family plus the geometry-only
  `plot_sky_track`;
- the **live dashboard** — `live_track`, a `matplotlib` animation over a rolling,
  now-centred `Trajectory` buffer.

As everywhere in propygator, parsing a TLE is pure-Python; the JVM spins up lazily only at
the first propagation (`docs/architecture.md` §10).

In [ ]:
import propygator as pgr

pgr.__version__

## 1. Get a TLE

Everything in 1.4 keys off a `TLE`. Fetch the current ISS element set from CelesTrak
(cached on disk), or fall back to pasted lines if you're offline — the rest of the
notebook runs either way.

In [ ]:
# A recent ISS element set, used as an offline fallback for the live fetch below.
ISS_LINE1 = "1 25544U 98067A   26171.41461525  .00008813  00000+0  16600-3 0  9990"
ISS_LINE2 = "2 25544  51.6327 284.1189 0004557 208.5194 151.5545 15.49333088572250"

try:
    tle = pgr.fetch_tle("ISS")  # also accepts a NORAD id, e.g. pgr.fetch_tle(25544)
    print("fetched live :", tle.name)
except pgr.TLEFetchError as err:
    print("offline - using a pinned ISS TLE.\n  ", err)
    tle = pgr.TLE.from_strings(ISS_LINE1, ISS_LINE2, name="ISS (ZARYA)")

## 2. Where is it *now*? — the realtime primitives

`current_state` and `current_ground_position` evaluate the TLE at `Epoch.now()`. They are
cheap one-shot queries — not a throwaway `Trajectory` — built on a shared single-shot SGP4
evaluation.

- `current_state(tle)` returns a `State` in **TEME**, SGP4's native frame, with no silent
  conversion (`docs/architecture.md` §10). Call `.to_frame(...)` to convert.
- `current_ground_position(tle)` returns a `GeodeticPosition` — lat/lon/alt directly. It
  carries no `Frame`, so it is allowed to convert internally (TEME→ITRF→geodetic).

In [ ]:
st = pgr.current_state(tle)
print("current_state frame :", st.frame)  # Frame.TEME
assert st.frame is pgr.Frame.TEME

gp = pgr.current_ground_position(tle)
print(
    f"sub-satellite point :"
    f"{gp.latitude_deg:6.2f} deg lat, "
    f"{gp.longitude_deg:7.2f} deg lon"
)
print(f"altitude            : {gp.altitude_m / 1e3:6.1f} km")

## 3. Look angles from a ground station

The observer-centric counterpart to `to_geodetic`: where in *my* sky is the satellite?
`look_angles(station, state)` returns an `AzElRange` — azimuth (0 = North, increasing
clockwise), elevation (0 = horizon, +90 = zenith; negative = below the horizon), and
slant range.

Like `to_geodetic`, the result carries no `Frame`, so `look_angles` converts its input
internally — you can hand it a TEME state directly.

In [ ]:
durham = pgr.GroundStation("Durham", 35.99, -78.90, altitude_m=130)

traj = pgr.propagate_tle(tle, duration=86400, output_step=60)  # one day in TEME
ae = pgr.look_angles(durham, traj[0])
print(
    f"az {ae.azimuth_deg:6.1f} deg   "
    f"el {ae.elevation_deg:6.1f} deg   "
    f"range {ae.range_m / 1e3:7.0f} km"
)

For a whole trajectory there is the batched `look_angles_track(station, trajectory)`
(one `TopocentricFrame`, parallel az/el/range arrays — reachable via
`propygator.core.observation`). The same topocentric kernel also projects the **Sun** and
**Moon**, which the live dashboard uses for its sky tint and markers — and which Feature
1.5 will reuse as its "is the observer in darkness?" gate.

In [ ]:
sun = pgr.sun_look_angles(durham, traj.start_epoch)
moon = pgr.moon_look_angles(durham, traj.start_epoch)
print(
    f"Sun  : az {sun.azimuth_deg:6.1f}  "
    f"el {sun.elevation_deg:6.1f}  ({'up' if sun.elevation_deg > 0 else 'down'})"
)
print(
    f"Moon : az {moon.azimuth_deg:6.1f}  "
    f"el {moon.elevation_deg:6.1f}  ({'up' if moon.elevation_deg > 0 else 'down'})"
)

## 4. The sky view — `plot_sky_track`

`plot_sky_track` draws the satellite's path across the observer's sky on a polar chart:
**North at top, azimuth clockwise, zenith at the centre and the horizon at the rim**
(radius = zenith angle). Samples below the horizon are lifted from the line, so each
visible pass draws as its own arc rather than a chord across the disk.

It is **geometry only** — no passes, eclipse shading, or brightness (those are Feature
1.5). The sky view is observer-relative az/el, independent of the trajectory's frame, so a
TEME trajectory is fine.

In [ ]:
pgr.plot_sky_track(traj, durham)

## 5. The live dashboard — `live_track`

`live_track` opens a live, self-updating view that tracks the satellite in real time. It
maintains a rolling `Trajectory` buffer **centred on now** (a trailing flown path + a
leading predicted path, with the live marker sliding between them) and drives a
`matplotlib.animation.FuncAnimation` over **three panels** — ground track, altitude, speed
— or **four** when you pass a `GroundStation` (adding the live sky view).

Two things to know:

- **It needs an interactive backend.** Run `%matplotlib widget` (in-notebook, via `ipympl`)
  or `%matplotlib qt` (a pop-out window) first. The default `%matplotlib inline` draws only
  a static snapshot — it does not animate.
- **Keep the returned reference.** Assign it (`anim = pgr.live_track(...)`); if the
  animation object is garbage-collected, matplotlib silently stops it.

The view is **display-only** — it is never saved. Passing a friendly **name** (e.g.
`"ISS"`) auto-refreshes the TLE from CelesTrak as the buffer rolls; passing a `TLE` object
re-propagates only.

In [ ]:
# Enable an interactive backend first, e.g.:
#   %matplotlib widget

# Pass the TLE we already have (re-propagates only). To auto-refresh from CelesTrak as
# the view runs, pass the name instead: pgr.live_track("ISS", durham).
anim = pgr.live_track(tle, durham)  # 4-panel; pgr.live_track(tle) for the 3-panel view
anim

---

That's Feature 1.4: **where is it now** (`current_state` / `current_ground_position`),
**where is it in my sky** (`look_angles` / `plot_sky_track`), and **show me live**
(`live_track`) — all composed over `propagate_tle` and Feature 1.1's output surface. The
full contract is in `docs/features.md` §1.4.